In this chapter, the architecture that we will replicate fully is GPT 2.

(Didn't we already do this with Kaparthy? Yes, so this is just kinda like rehab practice, really the entire book is just rehab practice.)

These are the configurations for the GPT 2 model.

In [46]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

- vocab_size: size of unique tokens, and their respective embeddings 
- context_length: the model's maximum input token count.
- emb_dim: size of embedding for the token inputs
- n_heads: number of heads for multi-head attention
- n_layers: number of transformer layers in the model
- drop_rate: dropout likehihood for the dropout mechanism
- qkv_bias: shouldn't be a parameter bro why would qkv ever have bias

Our final class would look something like this... except all the functions in it are currently placeholders.

In [9]:
import torch
import torch.nn as nn


class DumbGPT2(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout_emb = nn.Dropout(cfg["drop_rate"])

        self.trasformer_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.out = nn.Sequential(
            LayerNorm(cfg),
            nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        )

    def forward(self, word_idx):
        batch_size, seq_len = word_idx.shape
        tok_embeds = self.tok_emb(word_idx)
        pos_embeds = self.pos_emb(word_idx)

        x = self.dropout_emb(tok_embeds + pos_embeds)
        x = self.trasformer_blocks(x)
        x = self.out(x)
        return x # x is raw logits, of dimension...
        


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # A simple placeholder

    def forward(self, x):
        # This block does nothing and just returns its input.
        return x


class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # The parameters here are just to mimic the LayerNorm interface.

    def forward(self, x):
        # This layer does nothing and just returns its input.
        return x


The forward method describes the data flow through the model: it computes token and positional embeddings for the input indices, applies dropout, processes the data through the transformer blocks, applies normalization, and finally produces logits with the linear output layer.

So now lets just code all of this out, shall we?

## Layer Norm

One of the problems that a neural network encounters is exploding/vanishing gradients.

TLDR being how multiplying a number that's larger than 1 repeatedly would make a vlaue go to infinity.

The other of how repeatedly multiplying numbers smaller than 1 would make a value go to 0.



Normalization is about another problem of how the variance and mean of the random variable (which, is just the vector propagating through the network) becoming larger and larger.

Uncontrolled variance breaks three things at once:
- Numerical stability
- Nonlinearity behavior
- Gradient flow

<img src='attachments/Screenshot 2026-02-13 at 11.07.42 PM.png' width=500>

In [10]:
torch.manual_seed(42)

example_layer_values = torch.randn(2, 5)

lin = nn.Linear(5, 6)
relu = nn.ReLU()

with torch.no_grad():
    out = relu(lin(example_layer_values))
    print(out)
    print(out.shape)
    print()
    print(out.mean(dim=-1, keepdim=True))
    print(out.var(dim=-1, keepdim=True))

tensor([[0.0000, 0.1842, 0.0052, 0.7233, 0.0000, 0.5298],
        [0.0000, 0.0000, 0.0000, 0.2237, 0.0000, 0.7727]])
torch.Size([2, 6])

tensor([[0.2404],
        [0.1661]])
tensor([[0.0982],
        [0.0963]])


Layer normalization applies this normalization to the rows of the tensor independently.

<img src='attachments/Screenshot 2026-02-13 at 11.08.54 PM.png' width=500>

Now, how do you make that mean to be 0, and std to be 1?

This is back to old normal distributions... I forgot why at the moment, but you basically have this formula:

normed_value = (value - mean) / std(value)

In [11]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
out_norm = (out - mean) / torch.sqrt(var)
print("Normalized layer outputs:\n", out_norm)

norm_mean = out_norm.mean(dim=-1, keepdim=True)
norm_var = out_norm.var(dim=-1, keepdim=True)
print("Mean:\n", norm_mean)
print("Variance:\n", norm_var)

Normalized layer outputs:
 tensor([[-0.7672, -0.1794, -0.7506,  1.5410, -0.7672,  0.9234],
        [-0.5351, -0.5351, -0.5351,  0.1857, -0.5351,  1.9546]])
Mean:
 tensor([[0.0000e+00],
        [1.4901e-08]])
Variance:
 tensor([[1.0000],
        [1.0000]])


Now we can write this operation inside of a class.

In [12]:
class LayerNorm(nn.Module):

    def __init__(self):
        super().__init__()
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) # the eps is to make sure we don't square root 0

But! However, in the real LayerNorm, there are learnable parameters introduced in this function, which changes the class to be like.

In [13]:
class LayerNorm(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()
        self.eps = 1e-5
        self.gamma = nn.Parameter(torch.ones(hidden_dim))
        self.beta = nn.Parameter(torch.zeros(hidden_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * ((x - mean) / torch.sqrt(var + self.eps)) + self.beta # the eps is to make sure we don't square root 0

## GELU Activations

<img src='attachments/Screenshot 2026-02-14 at 9.04.06 AM.png' width=500>

It's just an activation function.

In [14]:
class GELU(nn.Module):

    def __init__(self):
        super().__init__()
        # there needs not to be parameters here

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))))

## Feed Forward

Again, this is nothing crazy, it's just a feed forward network in Pytorch.

However, the expanded the embedding dimension by 4 in the intermediate feed forward layer, and then reduced it down to the embedding dimension.

<img src='attachments/Screenshot 2026-02-14 at 10.01.39 AM.png' width=500>

In [15]:
class FeedForward(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)

## Residule Connections

Residule connections are a way to solve the vanishing gradient problem in deep neural networks.

It's like the addition of 2 spearetly pathways for gradients.

<img src='attachments/Screenshot 2026-02-14 at 10.06.56 AM.png' width=500>

This won't be implemented separetly, we can just implement it by ourselves in the trasnformer class

## Transformer Block

Now we will put everything together inside a transformer block, and we will use our own implemented multi-head attention from the previous chapter.

<img src='attachments/Screenshot 2026-02-14 at 10.42.35 AM.png' width=500>

In [ ]:
from math import sqrt


class MultAttn(nn.Module):

    def __init__(self, d_in=64, d_out=64, context_length=10, dropout=0.5, num_heads=8):
        super().__init__()
        self.w_q = nn.Linear(d_in, d_out, bias=False)
        self.w_k = nn.Linear(d_in, d_out, bias=False)
        self.w_v = nn.Linear(d_in, d_out, bias=False)
        self.head_dim = d_out // num_heads
        self.num_heads = num_heads

        self.dropout = nn.Dropout(dropout)
        self.mlp = nn.Linear(d_out, d_out)

        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):

        # x shape: [batch_size, num_tokens, d_in] d_in = emb_size
        b, num_tokens, d_in = x.shape

        # getting qkv
        q = self.w_q(x) # [batch_size, num_tokens, d_out]
        k = self.w_k(x)
        v = self.w_v(x)

        # reshape, multi-head attention
        # this didn't add anything new! We just 'split' the qkv (d_out) into 2 dims, (num_heads, head_size)
        q = q.view(b, num_tokens, self.num_heads, self.head_dim)
        k = k.view(b, num_tokens, self.num_heads, self.head_dim)
        v = v.view(b, num_tokens, self.num_heads, self.head_dim)

        # But we don't want the operations on num_heads! We want to do attention for each head respectively
        q = q.reshape(b, self.num_heads, num_tokens, self.head_dim)
        k = k.reshape(b, self.num_heads, num_tokens, self.head_dim)
        v = v.reshape(b, self.num_heads, num_tokens, self.head_dim)

        # doing attention
        attn_scores = q @ k.transpose(-2, -1) # [b, num_heads, num_tokens, num_tokens]

        # masking
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / sqrt(k.shape[-1]), dim=-1)

        # dropuot
        attn_weights = self.dropout(attn_weights)

        # combine heads after multiply with value
        # [b, num_heads, num_tokens, head_size] = [b, num_heads, num_tokens, num_tokens] * [b, num_heads, num_tokens, head_size] 
        context_vec = attn_weights @ v

        # [b, num_tokens, num_heads, head_size], this is for concatonation
        context_vec = context_vec.transpose(1, 2)

        # concat (which means reshaping here)
        # cintguous helps us change where the arrays are stored on a matrix, making it continous
        context_vec = context_vec.contiguous().view(b, num_tokens, d_in)
        context_vec = self.mlp(context_vec) # optional projection

        return context_vec



class LayerNorm(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()
        self.eps = 1e-5
        self.gamma = nn.Parameter(torch.ones(hidden_dim))
        self.beta = nn.Parameter(torch.zeros(hidden_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * ((x - mean) / torch.sqrt(var + self.eps)) + self.beta # the eps is to make sure we don't square root 0
    


class GELU(nn.Module):

    def __init__(self):
        super().__init__()
        # there needs not to be parameters here

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))))
    


class FeedForward(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)

Those are all the parts which we've made, now below is our final GPT block.

In [32]:
import torch
from torch import nn as nn


class TransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.attention = MultAttn(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"])
        
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["drop_rate"])


    def forward(self, x):
        
        # we implement residule connection here
        x_res = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + x_res

        x_res = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.dropout(x)
        x = x + x_res

        return x
        

Now let us try this model

In [33]:
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # Shape: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


## GPT architecture

now that we made a transformer block, what we should do is tie it together with the previous embeddings that we've made and complete the GPT model.

In [53]:
class GPT2(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout_emb = nn.Dropout(cfg["drop_rate"])

        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.out_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, x):

        # turn into tokens
        b, seq_len = x.shape
        token_emb = self.tok_emb(x)
        pos_emb = self.pos_emb(torch.arange(seq_len))
        x = self.dropout_emb(token_emb + pos_emb)

        # go through attention layers
        x = self.transformer_blocks(x)

        # final output layer
        x = self.out_norm(x)
        x = self.out_head(x)

        return x # this is logits, softmax is not applied yet

Now let's see actual text going into the model

In [51]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [55]:
torch.manual_seed(123)
model = GPT2(GPT_CONFIG_124M)

out = model(batch)
print("Input shape:\n", batch.shape)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input shape:
 torch.Size([2, 4])
Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.3504,  0.0623, -0.0723,  ..., -0.3020, -0.5341, -0.2686],
         [-0.0315, -0.9055, -0.8384,  ..., -0.7900, -0.3132,  0.0767],
         [ 0.8977,  0.1939, -0.6155,  ..., -0.3714, -1.2043, -0.0865],
         [-1.2214,  0.5001, -0.6910,  ...,  0.5042, -0.2890, -0.1869]],

        [[-0.7703,  0.2502,  0.2255,  ...,  0.0310, -0.7171, -0.8077],
         [ 0.2568, -0.0455,  0.3045,  ...,  1.0134, -0.5447,  0.7185],
         [ 0.8977,  0.6368, -0.0116,  ...,  0.3994, -0.6144, -0.0369],
         [-0.4131,  0.2752,  0.3734,  ...,  0.6909, -1.0071, -0.1509]]],
       grad_fn=<UnsafeViewBackward0>)


## Generating Text

Isn't this exiciting? Now we need to understand how to generate text actually using this model.

- We don't have a end of text token at the moment, so all that we can do is allow the model output tokens up to a certain length.

- And also, we are not doing sampling here from a probability distribution for the next possible token, we are simply dead picking the token with the highest value.

- This is also not meant to be worked with on a batch basis, we are allowing 1 string input and that's it.

<img src='attachments/Screenshot 2026-02-14 at 1.16.07 PM.png' width=500>


In [80]:
def encode(text, tokenizer):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0)

def decode(logits, tokenizer):
    # tokenizer doesn't take in torch tensor formats
    return tokenizer.decode(logits.squeeze(0).tolist())


# not very efficient but for demonstration purposes
def generate_text(text, tokenizer, model, max_len_new):
    
    idx = encode(text, tokenizer)  # keep token ids as tensor
    model.eval()

    for _ in range(max_len_new):
        with torch.no_grad():
            logits = model(idx)
            logits = logits[:, -1, :]
            next_idx = torch.argmax(logits, dim=-1, keepdim=True)
            idx = torch.cat((idx, next_idx), dim=1)

            print(f"Next token is: {decode(next_idx, tokenizer)} | value: {next_idx.item()}")
            print("Now token ids are:", idx.unsqueeze(0).tolist())
            print(f"The text is: {decode(idx, tokenizer)}")
            print()

    return decode(idx, tokenizer)


In [ ]:
start_context = "Hello, I am"
print(f"Starting text is: {start_context}")
print()
final_text = generate_text(start_context, tokenizer, model, max_len_new=5)

Starting text is: Hello, I am

Next token is: ages | value: 1095
Now token ids are: [[[15496, 11, 314, 716, 1095]]]
The text is: Hello, I amages

Next token is: iman | value: 24086
Now token ids are: [[[15496, 11, 314, 716, 1095, 24086]]]
The text is: Hello, I amagesiman

Next token is:  Bye | value: 47843
Now token ids are: [[[15496, 11, 314, 716, 1095, 24086, 47843]]]
The text is: Hello, I amagesiman Bye

Next token is: swick | value: 30961
Now token ids are: [[[15496, 11, 314, 716, 1095, 24086, 47843, 30961]]]
The text is: Hello, I amagesiman Byeswick

Next token is:  leave | value: 2666
Now token ids are: [[[15496, 11, 314, 716, 1095, 24086, 47843, 30961, 2666]]]
The text is: Hello, I amagesiman Byeswick leave



'Hello, I amagesiman Byeswick leave'

As you can see, this is not very intelligeble, as the model is made, but no learning is done.

Training will happen in the next chapter of the book.